In [1]:
%pwd


'/Users/wft08/Desktop/CHATBOTAI 2/medibot/research'

In [2]:
import os
os.chdir("../")


In [3]:

%pwd


'/Users/wft08/Desktop/CHATBOTAI 2/medibot'

In [4]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
def load_pdf_file(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)

    documents=loader.load()

    return documents
extracted_data=load_pdf_file(data="/Users/wft08/Desktop/CHATBOTAI 2/Data")
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
    text_chunks = text_splitter.split_documents(extracted_data)

    return text_chunks
text_chunks = text_split(extracted_data)
print("length of my chunk:", len(text_chunks))

length of my chunk: 12


In [5]:
extracted_data

[Document(metadata={'producer': 'macOS Version 14.6 (Build 23G80) Quartz PDFContext', 'creator': 'Adobe Illustrator 26.5 (Macintosh)', 'creationdate': '2025-05-08T05:37:57+00:00', 'creatorversion': '21.0.0', 'moddate': '2025-06-05T10:47:05+05:30', 'title': 'foursight-challenge-navigator', 'source': '/Users/wft08/Desktop/CHATBOTAI 2/Data/foursight.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Clarify\nGoal\nChallenge\n What do you want to accomplish? Finish the sentence, “It would be great if...”\nNow pinpoint t\nhe right challenge question. What question, if answered, would lead to a breakthrough? Review your data \nuntil you ﬁnd one that frames the right problem. Invite new thinking by beginning each question with a phrase like:\nPut a  by the question that points you in the direction of a breakthrough. Write it at the top of the next page.\n1. How to...?\n2. How might...?\n3. In what ways might...?\n4. What might be all the...?\n5.\n6.\n7.\n8.\n9.\n10.\nIt woul

In [6]:
from langchain.embeddings import HuggingFaceEmbeddings

In [7]:
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [8]:
embeddings = download_hugging_face_embeddings()

/var/folders/1p/rxdlwgxj1lb9j3q3x30x2l4c0000gn/T/ipykernel_2104/4238859041.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/Users/wft08/Desktop/CHATBOTAI 2/medibot/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [10]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
import os
PINECONE_API_KEY= os.environ.get("PINECONE_API_KEY")


In [12]:
from dotenv import load_dotenv
load_dotenv()

import os
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Step 1: Load API key
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")

# Step 2: Create Pinecone client instance (NO init)
pc = Pinecone(api_key=PINECONE_API_KEY)

# Step 3: Index name
index_name = "test"

# Step 4: (Optional) Create index if not exists
if index_name not in [index["name"] for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,  # Must match the embedding model's output dimension
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

# Step 5: Load PDF and split
loader = PyPDFLoader("/Users/wft08/Desktop/CHATBOTAI 2/Data/foursight.pdf")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
text_chunks = text_splitter.split_documents(documents)

# Step 6: Create embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Step 7: Store in Pinecone
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=index_name
)


In [13]:
from pinecone import Pinecone

In [14]:
query = "How to build good habits?"
results = docsearch.similarity_search(query, k=3)

for i, r in enumerate(results):
    print(f"\nResult {i+1}:\n{r.page_content}")



Result 1:
around,	build	a	life	around.
There	is	no	one	right	way	to	create	better	habits,	but	this	book	describes	the
best	way	I	know—an	approach	that	will	be	effective	regardless	of	where	you
start	or	what	you’re	trying	to	change.	The	strategies	I	cover	will	be	relevant	to
anyone	looking	for	a	step-by-step	system	for	improvement,	whether	your	goals
center	on	health,	money,	productivity,	relationships,	or	all	of	the	above.	As	long
as	human	behavior	is	involved,	this	book	will	be	your	guide.

Result 2:
around,	build	a	life	around.
There	is	no	one	right	way	to	create	better	habits,	but	this	book	describes	the
best	way	I	know—an	approach	that	will	be	effective	regardless	of	where	you
start	or	what	you’re	trying	to	change.	The	strategies	I	cover	will	be	relevant	to
anyone	looking	for	a	step-by-step	system	for	improvement,	whether	your	goals
center	on	health,	money,	productivity,	relationships,	or	all	of	the	above.	As	long
as	human	behavior	is	involved,	this	book	will	be	your	guide.

Resul

In [15]:
from dotenv import load_dotenv
load_dotenv()

import os
import pinecone
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.llms import CTransformers
from langchain_pinecone import PineconeVectorStore

llm = CTransformers(
    model="/Users/wft08/Desktop/CHATBOTAI 2/medibot/research/model/llama-2-7b-chat.ggmlv3.q4_0.bin",
    model_type="llama",
    config={
        "max_new_tokens": 56,           # Reduce token size to speed up
        "temperature": 0.3,
        "threads": 4,                    # Use 4 CPU threads if available
        "batch_size": 8,                 # Lower = less RAM needed
        "context_length": 2048          # Match model context (adjust if needed)
    }
)

print("Model loaded.")
print("Waiting for user input...")
print("Testing LLM response:")
response = llm("i am afraid to learn what should i do?")
print("Response test:", response)





Model loaded.
Waiting for user input...
Testing LLM response:


/var/folders/1p/rxdlwgxj1lb9j3q3x30x2l4c0000gn/T/ipykernel_2104/3505768248.py:29: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = llm("i am afraid to learn what should i do?")


Response test: 
 Unterscheidung between fear and anxiety 1. Fear is a normal human emotion that helps us recognize and avoid potential dangers. It can be triggered by a specific situation or object, such as a snake or heights. Anxiety, on the other hand


In [16]:
print("🔍 Testing LLM response:")
response = llm("how doing 1 percent  extra each day works?")
print("Response test:", response)

🔍 Testing LLM response:
Response test: 
 Unterscheidung between the two is important because it can help you understand how to use these strategies effectively.

The "1% rule" is a simple strategy that involves setting aside 1% of your income each day or week towards savings or investments. The idea


In [17]:
from langchain.prompts import PromptTemplate

template = """
You are a helpful assistant. Use the provided context to answer the question clearly and concisely in 1–2 sentences. Ensure your response ends with a complete sentence.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)


In [18]:
from langchain.chains import RetrievalQA
from langchain_pinecone import PineconeVectorStore
vectorstore = PineconeVectorStore.from_existing_index(
    index_name="medicalbot",
    embedding=embeddings
)

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt}
)


In [21]:
query = "what is next js ?"
response = qa_chain.run(query)

print("Response:\n", response)


Response:
 Next.js is a popular React-based framework for building server-side rendered (SSR) web applications. It provides a set of features and tools to simplify the development process, including automatic code splitting, built-in support for internationalization (i18n), and
